# Feature Engineering


In [1]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from olist import config as cfg, features, io_utils as io

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
cfg.ensure_dirs()

clean = io.load_parquet_dir(cfg.CLEAN_DIR)
orders = clean["olist_orders_dataset"]
items = clean["olist_order_items_dataset"]
payments = clean["olist_order_payments_dataset"]
reviews = clean["olist_order_reviews_dataset"]
customers = clean["olist_customers_dataset"]
products = clean["olist_products_dataset"]
sellers = clean["olist_sellers_dataset"]
geolocation = clean["olist_geolocation_dataset"]

print(f"loaded {len(clean)} clean tables")
print("orders:", orders.shape, "| items:", items.shape)

loaded 9 clean tables
orders: (99441, 12) | items: (112650, 12)


## 1. Order geography


In [2]:
order_geo = features.build_order_geo(orders, items, customers, sellers, geolocation)
print(order_geo.shape)
print("\ndistance coverage:", f"{order_geo['distance_km'].notna().mean():.2%}")
print(order_geo[["distance_km", "distance_km_mean", "n_shipping_legs", "same_state_flag"]].describe().round(1).to_string())
print("\ndistance buckets:")
print(order_geo["distance_bucket"].value_counts().sort_index().to_string())

(98666, 12)

distance coverage: 99.50%
       distance_km  distance_km_mean  n_shipping_legs  same_state_flag
count      98176.0           98176.0          98666.0          98666.0
mean         602.6             601.3              1.0              0.4
std          594.1             593.6              0.1              0.5
min            0.0               0.0              1.0              0.0
25%          188.3             186.5              1.0              0.0
50%          435.4             433.8              1.0              0.0
75%          800.7             797.9              1.0              1.0
max         3579.3            3579.3              5.0              1.0

distance buckets:
distance_bucket
1000km+       15793
200-500km     30902
50-200km      13112
500-1000km    26375
<50km         11994


In [3]:
# Sanity check the haversine implementation against a distance we know.
# Sao Paulo -> Rio de Janeiro is roughly 360 km.
print("SP -> RJ:", round(float(features.haversine_km(-23.55, -46.63, -22.91, -43.17)), 1), "km")
assert 330 < features.haversine_km(-23.55, -46.63, -22.91, -43.17) < 390

multi = order_geo[order_geo["n_shipping_legs"] > 1]
print(f"\nmulti-seller orders: {len(multi):,} ({len(multi) / len(order_geo):.2%})")
print("for these, max and mean distance genuinely differ:")
print(multi[["distance_km", "distance_km_mean"]].head().round(1).to_string(index=False))

SP -> RJ: 360.6 km

multi-seller orders: 1,278 (1.30%)
for these, max and mean distance genuinely differ:
 distance_km  distance_km_mean
      1310.1            1089.6
       950.4             903.8
      1011.0             784.7
       786.4             658.9
       424.3             389.9


## 2. Order features


In [4]:
order_features = features.build_order_features(
    orders, items, payments, products, order_geo, customers
)
print(order_features.shape)
assert order_features["Order_Id"].is_unique, "grain broken"
print("grain: one row per Order_Id  PASS")

(99441, 61)
grain: one row per Order_Id  PASS


In [5]:
known_at_purchase = [
    "order_item_count", "order_seller_count", "order_gmv", "order_freight_total",
    "avg_item_price", "freight_ratio", "freight_per_kg", "order_weight_g",
    "order_volume_cm3", "estimated_delivery_days", "shipping_limit_days",
    "distance_km", "same_state_flag", "purchase_hour", "purchase_weekday_num",
    "max_installments", "payment_count", "dominant_payment_type", "main_category",
]
realised_after = [
    "approval_hours", "carrier_hours", "last_mile_hours", "total_delivery_hours",
    "delivery_days", "delivery_delay_days", "late_delivery_flag", "cancelled_flag",
]

print("KNOWN AT PURCHASE - allowed as predictors")
print(order_features[known_at_purchase].describe(include="all").T[["count", "mean", "min", "max"]].to_string())
print("\n\nREALISED AFTER PURCHASE - targets and KPIs only")
print(order_features[realised_after].describe().T[["count", "mean", "min", "max"]].round(2).to_string())

KNOWN AT PURCHASE - allowed as predictors
                           count          mean       min          max
order_item_count         98666.0      1.141731       1.0         21.0
order_seller_count       98666.0      1.013622       1.0          5.0
order_gmv                98666.0    137.754076      0.85      13440.0
order_freight_total      98666.0     22.823562       0.0      1794.96
avg_item_price           98666.0    125.919255      0.85       6735.0
freight_ratio            98666.0      0.308389       0.0    21.447059
freight_per_kg           98660.0      39.81151       0.0      19800.0
order_weight_g           98666.0   2390.156893       0.0     184400.0
order_volume_cm3         98666.0  17402.597612     168.0    1476000.0
estimated_delivery_days  99441.0      23.76765  1.648993   155.135463
shipping_limit_days      98666.0      6.617315  2.004236  1056.041667
distance_km              98176.0    602.628656       0.0  3579.339684
same_state_flag          98666.0       0.35831  

## 3. Undelivered orders


In [6]:
undelivered = order_features["Order_Delivered_Customer_Date"].isna()
print(f"orders total          : {len(order_features):,}")
print(f"undelivered           : {undelivered.sum():,} ({undelivered.mean():.2%})")
print(f"late_delivery_flag NaN: {order_features['late_delivery_flag'].isna().sum():,}")

assert order_features.loc[undelivered, "late_delivery_flag"].isna().all()
print("\nPASS - every undelivered order is NaN, not 0.")

late_rate = order_features["late_delivery_flag"].mean()
naive_rate = order_features["late_delivery_flag"].fillna(0).mean()
print(f"\nhonest late rate (delivered orders only): {late_rate:.2%}")
print(f"if NaN were filled with 0               : {naive_rate:.2%}")
print(f"the shortcut understates the problem by  {(late_rate - naive_rate) / late_rate:.1%} relative")

orders total          : 99,441
undelivered           : 2,965 (2.98%)
late_delivery_flag NaN: 2,965

PASS - every undelivered order is NaN, not 0.

honest late rate (delivered orders only): 8.11%
if NaN were filled with 0               : 7.87%
the shortcut understates the problem by  3.0% relative


In [7]:
delay = order_features["delivery_delay_days"].dropna()
print("delivery delay vs promise, in days (negative = early):")
print(delay.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(1).to_string())
print(f"\nmedian order arrives {abs(delay.median()):.0f} days "
      f"{'early' if delay.median() < 0 else 'late'}")
print(f"but {(delay > 7).mean():.2%} of orders arrive more than a week late")
print("\n=> the promise is padded and the problem is the tail, not the centre.")

delivery delay vs promise, in days (negative = early):
count    96476.0
mean       -11.2
std         10.2
min       -146.0
5%         -25.9
25%        -16.2
50%        -11.9
75%         -6.4
95%          3.8
99%         18.9
max        189.0

median order arrives 12 days early
but 3.47% of orders arrive more than a week late

=> the promise is padded and the problem is the tail, not the centre.


## 4. Customer features


In [8]:
customer_features = features.build_customer_features(order_features)
product_features = features.build_product_features(items, products, order_features)
seller_features = features.build_seller_features(items, sellers, order_features)
review_features = features.build_review_features(reviews, order_features)

for name, df in [("customer_features", customer_features), ("product_features", product_features),
                 ("seller_features", seller_features), ("review_features", review_features)]:
    print(f"{name:<20} {df.shape}")

customer_features    (93358, 19)
product_features     (32951, 13)
seller_features      (3095, 12)
review_features      (98167, 19)


In [9]:
print("RFM segments:")
seg = (
    customer_features.groupby("rfm_segment_label")
    .agg(customers=("Customer_Unique_Id", "size"), value=("monetary_value", "sum"),
         avg_recency=("recency_days", "mean"), avg_freq=("frequency_orders", "mean"))
    .sort_values("value", ascending=False)
)
seg["customer_share_pct"] = (seg["customers"] / seg["customers"].sum() * 100).round(1)
seg["value_share_pct"] = (seg["value"] / seg["value"].sum() * 100).round(1)
print(seg.round(1).to_string())

one_timers = customer_features["frequency_orders"].eq(1).mean()
print(f"\ncustomers who bought exactly once: {one_timers:.1%}")
print("This is why RFM here is a diagnostic, not a retention programme:")
print("with ~3% repeat buyers, F and M carry almost no variance.")

RFM segments:
                   customers      value  avg_recency  avg_freq  customer_share_pct  value_share_pct
rfm_segment_label                                                                                  
At Risk                27846  5458668.5        443.6       1.0                29.8             35.4
Big Spenders            8983  2718380.4        140.6       1.0                 9.6             17.6
Champions              15009  2655895.8        139.0       1.1                16.1             17.2
Loyal                  15911  2121806.2        231.8       1.0                17.0             13.8
Others                 16379  1796717.4        198.1       1.0                17.5             11.7
Lost                    9230   670993.5        445.2       1.0                 9.9              4.4

customers who bought exactly once: 97.0%
This is why RFM here is a diagnostic, not a retention programme:
with ~3% repeat buyers, F and M carry almost no variance.


## 5. Seller reliability


In [10]:
seller_hist = features.as_of_reliability(order_features, items, key="Seller_Id", prefix="seller")
route_hist = features.as_of_route_reliability(order_features)
customer_hist = features.as_of_customer_history(order_features)

print("seller_hist  ", seller_hist.shape, list(seller_hist.columns))
print("route_hist   ", route_hist.shape, list(route_hist.columns))
print("customer_hist", customer_hist.shape, list(customer_hist.columns))

seller_hist   (98666, 3) ['Order_Id', 'seller_prior_orders', 'seller_prior_late_rate']
route_hist    (99441, 4) ['Order_Id', 'route', 'route_prior_orders', 'route_prior_late_rate']
customer_hist (99441, 4) ['Order_Id', 'customer_prior_orders', 'customer_prior_revenue', 'is_repeat_purchase']


In [11]:
# Proof 1: a seller's very first order must carry zero prior observations.
# A multi-seller order is jointly attributed to every seller involved, so its
# summed seller_prior_orders includes the OTHER sellers' history. Only enforce
# the no-leakage guarantee for true single-seller first orders.
link = items[["Order_Id", "Seller_Id"]].drop_duplicates()
seller_cnt = items.groupby("Order_Id")["Seller_Id"].nunique().rename("n_sellers")
first_order_per_seller = (
    order_features[["Order_Id", "Order_Purchase_Timestamp"]]
    .merge(link, on="Order_Id")
    .sort_values("Order_Purchase_Timestamp")
    .drop_duplicates(subset="Seller_Id", keep="first")
)
firsts = seller_hist[seller_hist["Order_Id"].isin(first_order_per_seller["Order_Id"])].merge(
    first_order_per_seller[["Order_Id", "Seller_Id"]], on="Order_Id"
).merge(seller_cnt, on="Order_Id")

single = firsts[firsts["n_sellers"] == 1]
multi = firsts[firsts["n_sellers"] > 1]
print(f"single-seller first orders checked : {len(single):,}")
print(f"multi-seller first orders excluded  : {len(multi):,}")
print(f"max prior_orders among single       : {single['seller_prior_orders'].max():.0f}")
assert len(single) == 0 or (single["seller_prior_orders"] == 0).all()
print("PASS - no single-seller order can see its own seller's future (multi-seller excluded).\n")

# Proof 2: contrast with the naive lifetime rate.
naive = (
    items.merge(order_features[["Order_Id", "late_delivery_flag"]], on="Order_Id")
    .groupby("Seller_Id")["late_delivery_flag"].mean().rename("naive_lifetime_rate")
)
compare = (
    seller_hist.merge(link, on="Order_Id")
    .merge(naive, on="Seller_Id")
    .merge(order_features[["Order_Id", "Order_Purchase_Timestamp"]], on="Order_Id")
    .sort_values("Order_Purchase_Timestamp")
)
print("early orders - as-of rate knows almost nothing, naive rate already 'knows' everything:")
preview = compare.head(6)[["Order_Purchase_Timestamp", "seller_prior_orders",
                           "seller_prior_late_rate", "naive_lifetime_rate"]].copy()
preview["Order_Purchase_Timestamp"] = preview["Order_Purchase_Timestamp"].dt.date
print(preview.round(3).to_string(index=False))


single-seller first orders checked : 3,006
multi-seller first orders excluded  : 89
max prior_orders among single       : 0
PASS - no single-seller order can see its own seller's future (multi-seller excluded).

early orders - as-of rate knows almost nothing, naive rate already 'knows' everything:
Order_Purchase_Timestamp  seller_prior_orders  seller_prior_late_rate  naive_lifetime_rate
              2016-09-04                  0.0                   0.081                0.071
              2016-09-05                  0.0                   0.081                0.000
              2016-09-15                  0.0                   0.081                0.250
              2016-10-02                  0.0                   0.081                0.052
              2016-10-03                  0.0                   0.081                0.101
              2016-10-03                  0.0                   0.081                0.000


In [12]:
# Proof 3: the as-of rate converges on the lifetime rate as evidence accumulates.
buckets = pd.cut(compare["seller_prior_orders"], [-1, 0, 5, 25, 100, np.inf],
                 labels=["0", "1-5", "6-25", "26-100", "100+"])
convergence = compare.groupby(buckets, observed=True).agg(
    orders=("Order_Id", "size"),
    mean_as_of_rate=("seller_prior_late_rate", "mean"),
    mean_naive_rate=("naive_lifetime_rate", "mean"),
)
convergence["gap"] = (convergence["mean_as_of_rate"] - convergence["mean_naive_rate"]).round(4)
print(convergence.round(4).to_string())
print("\nThe gap shrinks as prior evidence grows - exactly what a shrunk as-of estimate should do.")

                     orders  mean_as_of_rate  mean_naive_rate     gap
seller_prior_orders                                                  
0                      5522           0.0811           0.0886 -0.0075
1-5                   11099           0.0762           0.0817 -0.0055
6-25                  19687           0.0666           0.0782 -0.0116
26-100                26825           0.0603           0.0760 -0.0157
100+                  36877           0.0652           0.0801 -0.0149

The gap shrinks as prior evidence grows - exactly what a shrunk as-of estimate should do.


## 6. Model base table


In [13]:
model_base = features.build_model_base(order_features, items)
print(model_base.shape)
print("\nsplit boundaries: train <", cfg.TRAIN_END.date(), "| valid <", cfg.VALID_END.date(), "| test after")
split_summary = (
    model_base.groupby("split")
    .agg(orders=("Order_Id", "size"),
         first_purchase=("Order_Purchase_Timestamp", "min"),
         last_purchase=("Order_Purchase_Timestamp", "max"),
         late_rate=("late_delivery_flag", "mean"))
    .reindex(["train", "valid", "test"])
)
split_summary["late_rate"] = split_summary["late_rate"].round(4)
print(split_summary.to_string())

(99441, 48)

split boundaries: train < 2018-05-01 | valid < 2018-07-01 | test after
       orders      first_purchase       last_purchase  late_rate
split                                                           
train   73577 2016-09-04 21:15:19 2018-04-30 23:47:26     0.0879
valid   13040 2018-05-01 00:02:11 2018-06-30 23:59:49     0.0497
test    12824 2018-07-01 00:25:07 2018-10-17 17:30:18     0.0748


In [14]:
# The splits must not overlap in time. This is the whole point of the design.
bounds = model_base.groupby("split")["Order_Purchase_Timestamp"].agg(["min", "max"])
assert bounds.loc["train", "max"] < bounds.loc["valid", "min"]
assert bounds.loc["valid", "max"] < bounds.loc["test", "min"]
print("PASS - train, valid and test are strictly ordered in time, no overlap.")

print(f"\nfeature contract: {len(features.PURCHASE_TIME_FEATURES)} predictors")
for name in features.PURCHASE_TIME_FEATURES:
    marker = "ok " if name in model_base.columns else "MISSING"
    print(f"  [{marker}] {name}")

PASS - train, valid and test are strictly ordered in time, no overlap.

feature contract: 39 predictors
  [ok ] order_item_count
  [ok ] order_product_count
  [ok ] order_seller_count
  [ok ] order_gmv
  [ok ] avg_item_price
  [ok ] max_item_price
  [ok ] is_multi_seller
  [ok ] order_weight_g
  [ok ] order_volume_cm3
  [ok ] max_item_weight_g
  [ok ] order_freight_total
  [ok ] freight_ratio
  [ok ] freight_per_kg
  [ok ] order_total_paid
  [ok ] payment_count
  [ok ] max_installments
  [ok ] estimated_delivery_days
  [ok ] shipping_limit_days
  [ok ] purchase_month_num
  [ok ] purchase_weekday_num
  [ok ] purchase_hour
  [ok ] purchase_is_weekend
  [ok ] days_since_dataset_start
  [ok ] distance_km
  [ok ] distance_km_mean
  [ok ] same_state_flag
  [ok ] n_shipping_legs
  [ok ] seller_prior_orders
  [ok ] seller_prior_late_rate
  [ok ] route_prior_orders
  [ok ] route_prior_late_rate
  [ok ] customer_prior_orders
  [ok ] customer_prior_revenue
  [ok ] is_repeat_purchase
  [ok ] domin

In [15]:
# Class balance drifts across the splits. Worth knowing before reading any metric.
print("late rate by month (delivered orders only):")
by_month = (
    model_base.assign(month=model_base["Order_Purchase_Timestamp"].dt.to_period("M"))
    .groupby("month")
    .agg(orders=("Order_Id", "size"), late_rate=("late_delivery_flag", "mean"))
)
by_month["late_rate"] = by_month["late_rate"].round(4)
print(by_month.tail(12).to_string())
print("\nThe target rate is not stationary. A chronological test set is the only")
print("split that exposes this; a random split would average it away.")

late rate by month (delivered orders only):
         orders  late_rate
month                     
2017-11    7544     0.1431
2017-12    5673     0.0838
2018-01    7269     0.0656
2018-02    6728     0.1600
2018-03    7211     0.2136
2018-04    6939     0.0531
2018-05    6873     0.0824
2018-06    6167     0.0136
2018-07    6292     0.0448
2018-08    6512     0.1039
2018-09      16        NaN
2018-10       4        NaN

The target rate is not stationary. A chronological test set is the only
split that exposes this; a random split would average it away.


## 7. Feature catalogue


In [16]:
CATALOGUE = [
    dict(name="order_gmv", formula="sum(Price) per Order_Id", grain="order",
         known_at="purchase", use="BI revenue proxy + predictor",
         nulls="0 if the order has no item rows", leakage="none",
         validation=">= 0; reconciles to items.Price total"),
    dict(name="order_freight_total", formula="sum(Freight_Value) per Order_Id", grain="order",
         known_at="purchase", use="predictor, and target of project 03",
         nulls="0 if no items", leakage="none",
         validation="reconciles to items.Freight_Value total"),
    dict(name="freight_ratio", formula="order_freight_total / order_gmv", grain="order",
         known_at="purchase", use="logistics cost efficiency",
         nulls="NaN when gmv = 0 (guarded)", leakage="none",
         validation="0 <= ratio, extreme values reviewed not capped"),
    dict(name="estimated_delivery_days", formula="Order_Estimated_Delivery_Date - Order_Purchase_Timestamp",
         grain="order", known_at="purchase", use="strongest predictor of lateness",
         nulls="none", leakage="none - the promise is made at checkout",
         validation="> 0 for every order"),
    dict(name="distance_km", formula="haversine(customer zip centroid, seller zip centroid), max over legs",
         grain="order", known_at="purchase", use="predictor for delay and freight",
         nulls="NaN when a zip prefix is absent from geolocation", leakage="none",
         validation="SP-RJ sanity check ~360 km"),
    dict(name="seller_prior_late_rate",
         formula="as-of shrunk mean of late_delivery_flag over deliveries COMPLETED before this purchase",
         grain="order", known_at="purchase",
         use="seller reliability predictor",
         nulls="none - falls back to the global rate via 20 pseudo-observations",
         leakage="none by construction; merge_asof on delivery time",
         validation="first order per seller has prior_orders = 0; rate in [0, 1]"),
    dict(name="route_prior_late_rate", formula="same as-of logic keyed on seller_state -> customer_state",
         grain="order", known_at="purchase", use="corridor reliability predictor",
         nulls="none - shrunk to global with 30 pseudo-observations", leakage="none by construction",
         validation="rate in [0, 1]"),
    dict(name="customer_prior_orders", formula="count of this customer's earlier orders",
         grain="order", known_at="purchase", use="repeat-buyer predictor",
         nulls="0 for a first order", leakage="none - cumcount excludes the current row",
         validation="first order per customer = 0; monotonic per customer"),
    dict(name="delivery_days", formula="(Order_Delivered_Customer_Date - Order_Purchase_Timestamp) / 24h",
         grain="order", known_at="AFTER DELIVERY", use="TARGET of project 02",
         nulls="NaN when undelivered", leakage="HIGH - never a predictor",
         validation="> 0; NaN only for undelivered"),
    dict(name="delivery_delay_days", formula="delivered - estimated, in days", grain="order",
         known_at="AFTER DELIVERY", use="KPI and target definition",
         nulls="NaN when undelivered", leakage="HIGH - never a predictor",
         validation="NaN exactly for undelivered orders"),
    dict(name="late_delivery_flag", formula="1 if delivery_delay_days > 0 else 0, NaN if undelivered",
         grain="order", known_at="AFTER DELIVERY", use="TARGET of project 01",
         nulls="NaN when undelivered - NEVER filled with 0", leakage="HIGH - never a predictor",
         validation="values in {0, 1, NaN}"),
    dict(name="recency_days", formula="SNAPSHOT_DATE - last purchase", grain="customer_unique_id",
         known_at="snapshot (fixed 2018-10-18)", use="RFM",
         nulls="none by construction", leakage="low for BI, HIGH for time-split ML",
         validation=">= 0; snapshot is a constant, never now()"),
    dict(name="seller_late_rate", formula="lifetime mean(late_delivery_flag) per Seller_Id", grain="seller",
         known_at="end of data", use="BI scorecard ONLY",
         nulls="NaN for sellers with no delivered order",
         leakage="HIGH if used as a model input - use seller_prior_late_rate instead",
         validation="in [0, 1]"),
    dict(name="rfm_segment_label", formula="rules over quintile r/f/m scores", grain="customer_unique_id",
         known_at="snapshot", use="BI segmentation",
         nulls="none", leakage="low",
         validation="one of Champions / Big Spenders / Loyal / At Risk / Lost / Others"),
]

catalogue = pd.DataFrame(CATALOGUE)
out = cfg.REPORTS_DIR / "feature_catalogue.md"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(
    "# Feature catalogue\n\n"
    "Generated by `notebooks/04_feature_engineering.ipynb`. Do not edit by hand.\n\n"
    "`known_at` is the field to read first: anything marked *AFTER DELIVERY* is a\n"
    "target or a KPI and must never appear in a feature matrix.\n\n"
    + catalogue.to_markdown(index=False) + "\n",
    encoding="utf-8",
)
print(f"wrote {out}")
catalogue[["name", "grain", "known_at", "leakage"]]

wrote C:\Users\programmer\Desktop\program\ali-ajvand-data-portfolio\olist-delivery-analytics\reports\feature_catalogue.md


,name,grain,known_at,leakage
0,order_gmv,order,purchase,none
1,order_freight_total,order,purchase,none
2,freight_ratio,order,purchase,none
3,estimated_delivery_days,order,purchase,none - the promise is made at checkout
4,distance_km,order,purchase,none
5,seller_prior_late_rate,order,purchase,none by construction; merge_asof on delivery time
6,route_prior_late_rate,order,purchase,none by construction
7,customer_prior_orders,order,purchase,none - cumcount excludes the current row
8,delivery_days,order,AFTER DELIVERY,HIGH - never a predictor
9,delivery_delay_days,order,AFTER DELIVERY,HIGH - never a predictor


## 8. Assemble feature tables


In [17]:
feature_tables = {
    "order_features": order_features,
    "order_geo_features": order_geo,
    "customer_features": customer_features,
    "customer_order_history_as_of": customer_hist,
    "product_features": product_features,
    "seller_features": seller_features,
    "review_features": review_features,
    "order_model_base": model_base,
}

grain = features.grain_audit(feature_tables)
recon = features.feature_reconciliation(clean, feature_tables)
print(grain.to_string(index=False))
print()
print(recon.to_string(index=False))

assert (grain["status"] == "PASS").all(), "a join fanned out - fix before modelling"
assert (recon["status"] == "PASS").all(), "totals drifted - fix before modelling"
print("\nGATE PASSED - grain intact, totals reconcile.")

                       table     declared_grain  rows  unique_keys status
              order_features           Order_Id 99441        99441   PASS
          order_geo_features           Order_Id 98666        98666   PASS
           customer_features Customer_Unique_Id 93358        93358   PASS
customer_order_history_as_of           Order_Id 99441        99441   PASS
            product_features         Product_Id 32951        32951   PASS
             seller_features          Seller_Id  3095         3095   PASS
             review_features           Order_Id 98167        98167   PASS
            order_model_base           Order_Id 99441        99441   PASS

          metric  clean_tables  feature_tables  diff  diff_pct status
   unique orders      99441.00        99441.00   0.0       0.0   PASS
item price total   13591643.70     13591643.70   0.0       0.0   PASS
   freight total    2251909.54      2251909.54   0.0       0.0   PASS
           items     112650.00       112650.00   0.0 

## 9. Save features


In [18]:
for name, df in feature_tables.items():
    target_dir = cfg.ML_DIR if name == "order_model_base" else cfg.FEATURES_DIR
    path = io.save_parquet(df, target_dir, name)
    print(f"{path.parent.name}/{path.name:<38} {df.shape}")

features/order_features.parquet                 (99441, 61)
features/order_geo_features.parquet             (98666, 12)
features/customer_features.parquet              (93358, 19)
features/customer_order_history_as_of.parquet   (99441, 4)
features/product_features.parquet               (32951, 13)
features/seller_features.parquet                (3095, 12)
features/review_features.parquet                (98167, 19)
ml/order_model_base.parquet               (99441, 48)


## 10. Summary
